In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
  
# metadata 
print(diabetes_130_us_hospitals_for_years_1999_2008.metadata) 
  
# variable information 
print(diabetes_130_us_hospitals_for_years_1999_2008.variables) 


In [ ]:
#import needed tools/libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)

seed = 123
np.random.seed(seed)

In [ ]:
# clean the data

X = X.replace("?", np.nan)
X = X.fillna("missing")

X = X.drop(
    columns=["encounter_id", 
             "patient_nbr", 
             "weight", 
             "payer_code", 
             "medical_specialty"],
            errors="ignore"
)

# make target binary
y = y["readmitted"]
y = (y == "<30").astype(int)

# encode categorical columns
X = pd.get_dummies(X, drop_first=True)

# train / validation / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=seed,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=seed,
    stratify=y_train
)

In [ ]:
# scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# logistic regression
# hyperparameter search: try C values
cs = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]

# initial values before test
best_c = None
best_val_auc = 0.0

for c in cs:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=seed)
    lr.fit(X_train_sc, y_train)
    
    # get predicted probabilities for validation set
    val_probs = lr.predict_proba(X_val_sc)

    # keep only probability of class 1
    val_probs_class1 = val_probs[:, 1]

# calculate validation AUC
    val_auc = roc_auc_score(y_val, val_probs_class1)

    print(f"C={c}, AUC={val_auc}")
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_c = c

print(f"\nBest C={best_c}  (AUC={best_val_auc})")

In [ ]:
# re-train best log regression and evaluate on test set
lr_best = LogisticRegression(C=best_c, max_iter=1000, random_state=seed)
lr_best.fit(X_train_sc, y_train)

lr_test_acc = accuracy_score(y_test, lr_best.predict(X_test_sc))
lr_test_auc = roc_auc_score(y_test, lr_best.predict_proba(X_test_sc)[:, 1])

print(f"Logistic Regression (C={best_c}), Test Accuracy: {lr_test_acc}, Test AUC: {lr_test_auc}")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, lr_best.predict(X_test_sc))).plot()
plt.title(f"Logistic Regression (C={best_c}) Confusion Matrix")
plt.show()

In [ ]:
# tentative
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

cs = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
train_aucs, val_aucs = [], []

for c in cs:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=seed)
    lr.fit(X_train_sc, y_train)
    train_aucs.append(roc_auc_score(y_train, lr.predict_proba(X_train_sc)[:, 1]))
    val_aucs.append(roc_auc_score(y_val,   lr.predict_proba(X_val_sc)[:, 1]))

plt.figure()
plt.semilogx(cs, train_aucs, marker='o', label="Train AUC")
plt.semilogx(cs, val_aucs,   marker='o', label="Val AUC")
plt.xlabel("C (regularization)")
plt.ylabel("AUC")
plt.title("Regularization Tradeoff: C vs AUC")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

# stronger regularization
cs = [0.0001, 0.001, 0.01, 0.1, 1, 10]
accs = []
recalls = [] 
precs = []

for c in cs:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=seed)
    lr.fit(X_train_sc, y_train)
    pred = lr.predict(X_val_sc)
    accs.append(accuracy_score(y_val, pred))
    recalls.append(recall_score(y_val, pred))
    precs.append(precision_score(y_val, pred))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

metrics = [accs, recalls, precs]
names = ["Accuracy", "Recall", "Precision"]

for i in range(len(names)):
    axes[i].semilogx(cs, metrics[i], marker='o')
    axes[i].set_title(names[i])
    axes[i].set_xlabel("C (regularization)")
    
plt.tight_layout()

In [ ]:
# notes: seems that logistic regression was relatively insensitive to regularization over a wide range of C values
# after C ≈ 0.1, performance metrics changed very little
# potential causes -> severe class imbalance, weak linear separability of readmitted versus non-readmitted patients?

In [ ]:
print(f"Logistic Regression (C={best_c})")
print(f"test acc: {lr_test_acc}, test AUC: {lr_test_auc}")

In [ ]:
# Accuracy tells how often the final predictions were correct,
#  while AUC tells how well the model separates readmitted vs non-readmitted patients. 
# Since the dataset is imbalanced, 
# AUC is more informative than accuracy.

In [ ]:
import matplotlib.pyplot as plt

# original target variable
readmission_counts = diabetes_130_us_hospitals_for_years_1999_2008.data.targets["readmitted"].value_counts()

print(readmission_counts)

plt.figure(figsize=(6,4))
readmission_counts.plot(kind="bar")
plt.title("Hospital Readmission Outcomes")
plt.xlabel("Readmission Status")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()